In [1]:
import pandas as pd
import os, json, librosa
import soundfile as sf
import io

In [2]:
df = pd.read_parquet(r'/home/nextai/asr-nepali/Nepali-ASR-FineTuning/rishi70612_audio_data.parquet')
print(df.columns)
print(df.iloc[0])

Index(['audio', 'sentence', 'path', 'input_ids', 'labels', 'input_length',
       'speaker_embeddings'],
      dtype='str')
audio                 {'bytes': b'RIFFz\x8b\x01\x00WAVEfmt \x10\x00\...
sentence                             पञ्चकुला भारतका सहरहरू एउटा सहर हो
path                  E:/.awajai/data/slr43/ne_np_female/wavs/nep_05...
input_ids             [151, 83, 91, 110, 152, 136, 150, 137, 107, 13...
labels                [[-3.3857956, -3.1920063, -3.1041048, -2.97389...
input_length                                                      50603
speaker_embeddings    [-0.08135123, -0.022780364, 0.039687287, 0.035...
Name: 0, dtype: object


input_ids→ tokens created for another model,
labels→ likely TTS training data,
input_length→ related to previous processing,
speaker_embedding→ information about the speaker's voice,
path→ old Windows path, probably unusable in Colab

In [3]:
for i in range(10):
    audio_bytes = df.iloc[i]['audio']['bytes']
    data, sr = sf.read(io.BytesIO(audio_bytes))
    print(i, sr) # Checking whether the audio frequency is 16khz or not

0 16000
1 16000
2 16000
3 16000
4 16000
5 16000
6 16000
7 16000
8 16000
9 16000


In [4]:
# Only these two columns matter
df = df[['audio', 'sentence']]

In [14]:
audio_dir = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio'
os.makedirs(audio_dir, exist_ok=True)
manifest_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as f:
    for i, row in df.iterrows():
        audio_bytes = row['audio']['bytes']
        wav_path = f"{audio_dir}/{i}.wav"
        data, sr = sf.read(io.BytesIO(audio_bytes))

        if sr != 16000:
          data = librosa.resample(data, orig_sr=sr, target_sr=16000)
          sr = 16000

        sf.write(wav_path, data, sr)
        duration = len(data) / sr

        entry = {
            "audio_filepath": wav_path,
            "text": row['sentence'],
            "duration": duration,
            "target_lang": "ne-NP"
        }
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [15]:
with open(manifest_path) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/0.wav", "text": "पञ्चकुला भारतका सहरहरू एउटा सहर हो", "duration": 3.1626875, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/1.wav", "text": "यो मठ तिब्‍बती बौद्ध धर्मका जेलुपा सम्प्रदायसँग सम्बन्धित‍ छ", "duration": 5.1706875, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/2.wav", "text": "ग्राहम बेलको जन्म स्कटल्यान्डको एन्डिनबर्गमा सन् तीन मार्च सन सत्र सय सत्चालिसमा भएको थियो", "duration": 7.8756875, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/3.wav", "text": "उहाँलाई चमत्कारिक व्यक्तित्वको रूपमा लिने गरिन्छ", "duration": 3.745, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/4.wav", "text": "कालो तिल तिलबाट प्राप्त हुने बिउ तेल 

In [16]:
import random

with open(manifest_path, encoding='utf-8') as f:
    lines = f.readlines()

random.seed(42)
random.shuffle(lines)

split_idx = int(len(lines) * 0.9)  # 90% train, 10% val
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

print(f"Total: {len(lines)}, Train: {len(train_lines)}, Val: {len(val_lines)}")

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(train_lines)

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(val_lines)

Total: 980, Train: 882, Val: 98


In [17]:
with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/489.wav", "text": "युनाइटेड पार्सल सर्भिस पचास हजार कामदार परिचालन गरेको छ", "duration": 4.036375, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/560.wav", "text": "जर्मन राष्ट्रिय फुटबल टोली र पूर्व फुटबल खेलाडीको प्रबन्धक", "duration": 4.956375, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/554.wav", "text": "फेसन पारखी उनी फेसनको क्षेत्रमा एक आइकन थिए", "duration": 4.085, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/832.wav", "text": "स्पेन पोर्चुगल पर्सियाका केही क्षेत्रहरूमा स्थापित इस्लाम धर्म खिलाफतको थालनी सय छ सय चालीस ई ताका भएको थियो", "duration": 8.612375, "target_lang": "ne-NP"}
{"audio_filepath": "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio/177.wav", "text":

In [10]:
import nemo
import nemo.collections.asr as nemo_asr
print(nemo.__version__)

/home/nextai/asr-nepali/Nepali-ASR-FineTuning/.myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: partially initialized module 'lightning.fabric' has no attribute 'strategies' (most likely due to a circular import)

In [ ]:
# !git clone https://github.com/NVIDIA/NeMo.git

In [ ]:
# !ls "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr"

In [11]:
from nemo.collections.asr.models import ASRModel

model_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

model = ASRModel.restore_from(model_path)


AttributeError: partially initialized module 'lightning.fabric' has no attribute 'strategies' (most likely due to a circular import)

In [15]:
!tar -tf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

427ad33c6285472cb01c3eb843d2309d_tokenizer.model
c8ead90c911846569df4738620acfc0f_vocab.txt
model_config.yaml
model_weights.ckpt


In [22]:
!mkdir -p /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer
!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "427ad33c6285472cb01c3eb843d2309d_tokenizer.model"
!mv "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/427ad33c6285472cb01c3eb843d2309d_tokenizer.model" \
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/tokenizer.model"
!ls -la /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer

!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "c8ead90c911846569df4738620acfc0f_vocab.txt"
!mv "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/c8ead90c911846569df4738620acfc0f_vocab.txt" \
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/vocab.txt"
!ls -la /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer


total 704
drwxrwxr-x 2 nextai nextai   4096 Jul 15 09:15 .
drwxrwxr-x 9 nextai nextai   4096 Jul 14 10:55 ..
-rw-r--r-- 1 nextai nextai 217588 May 29 23:18 model_config.yaml
-rw-r--r-- 1 nextai nextai 406554 Feb  3 18:15 tokenizer.model
-rw-r--r-- 1 nextai nextai  78294 Feb  3 18:20 vocab.txt
total 704
drwxrwxr-x 2 nextai nextai   4096 Jul 15 09:15 .
drwxrwxr-x 9 nextai nextai   4096 Jul 14 10:55 ..
-rw-r--r-- 1 nextai nextai 217588 May 29 23:18 model_config.yaml
-rw-r--r-- 1 nextai nextai 406554 Feb  3 18:15 tokenizer.model
-rw-r--r-- 1 nextai nextai  78294 Feb  3 18:20 vocab.txt


In [23]:
print(model_cfg['tokenizer'])

{'dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer', 'type': 'bpe'}


In [9]:
!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "model_config.yaml"

In [ ]:
import yaml

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)
model_cfg['tokenizer'] = {
    'dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer',
    'type': 'bpe'
}

model_cfg['train_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json' # Point data at manifests
model_cfg['train_ds']['is_tarred'] = False   # using a plain manifest, not tarred shards
model_cfg['train_ds']['shuffle']=True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['validation_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'
model_cfg['test_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json' # It exists because the model configuration expects a test dataset configuration, even if you don't use it during training.
model_cfg["optim"]["lr"] = 0.0001
model_cfg['validation_ds']['default_prompt_mode'] = 'unified'
model_cfg['validation_ds']['unified_auto_ratio'] = 0.0
model_cfg['validation_ds']['lang_field'] = 'target_lang'
model_cfg['test_ds']['default_prompt_mode'] = 'unified'
model_cfg['test_ds']['unified_auto_ratio'] = 0.0
model_cfg['test_ds']['lang_field'] = 'target_lang'
model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False # Hello
full_cfg = {
    'name': 'FastConformer-Transducer-BPE-Prompt-Streaming-Pilot',
    'model': model_cfg,
    'trainer': {
        'devices': 1, 'num_nodes': 1, 'max_epochs': -1,   # moot with Lhotse; -1 avoids implying it matters
        'accelerator': 'auto', 'precision': 'fp32',
        'log_every_n_steps': 10, 'enable_progress_bar': True,
        'num_sanity_val_steps': 0, 'enable_checkpointing': False,
        'logger': False, 'use_distributed_sampler': False,
        'max_steps': 250,
        'limit_train_batches': 50,   # ~5 pseudo-epochs over 250 steps -> 5 validation/checkpoint points
},
    'exp_manager': {
        'exp_dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run',
        'create_tensorboard_logger': True,
        'create_checkpoint_callback': True,
        'checkpoint_callback_params': {'monitor': 'val_wer', 'mode': 'min', 'save_top_k': 1, 'always_save_nemo': True},
        'resume_if_exists': False,
        'resume_ignore_no_checkpoint':True,
    },
    'init_from_nemo_model': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo',
}

out_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(full_cfg, f, allow_unicode=True, sort_keys=False)

print("Saved:", out_path)

Saved: /home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml


In [3]:
import yaml

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokenizer'] = {
    'dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer',
    'type': 'bpe'
}

# Data
model_cfg['train_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json'
model_cfg['train_ds']['is_tarred'] = False
model_cfg['train_ds']['shuffle'] = True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['validation_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'
model_cfg['test_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'

# Prompt/language routing
model_cfg['validation_ds']['default_prompt_mode'] = 'unified'
model_cfg['validation_ds']['unified_auto_ratio'] = 0.0
model_cfg['validation_ds']['lang_field'] = 'target_lang'
model_cfg['test_ds']['default_prompt_mode'] = 'unified'
model_cfg['test_ds']['unified_auto_ratio'] = 0.0
model_cfg['test_ds']['lang_field'] = 'target_lang'

model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False

# --- Optimizer: replaces flat lr=0.0001 with warmup + a real peak LR ---
# NoamAnnealing needs d_model to scale correctly — pull it from the encoder
# config rather than guessing, since a wrong value defeats the point of Noam.
d_model = model_cfg.get('encoder', {}).get('d_model')

if d_model:
    model_cfg['optim'] = {
        'name': 'adamw',
        'lr': 0.005,              # peak LR, only reached after warmup ramp
        'betas': [0.9, 0.98],
        'weight_decay': 1e-3,
        'sched': {
            'name': 'NoamAnnealing',
            'd_model': d_model,
            'warmup_steps': 200,
            'min_lr': 1e-6,
        },
    }
else:
    # Fallback: CosineAnnealing doesn't need d_model, safer if the encoder
    # config doesn't expose it under this key.
    model_cfg['optim'] = {
        'name': 'adamw',
        'lr': 0.005,
        'betas': [0.9, 0.98],
        'weight_decay': 1e-3,
        'sched': {
            'name': 'CosineAnnealing',
            'warmup_steps': 200,
            'min_lr': 1e-6,
        },
    }

full_cfg = {
    'name': 'FastConformer-Transducer-BPE-Prompt-Streaming-Pilot',
    'model': model_cfg,
    'trainer': {
        'devices': 1, 'num_nodes': 1, 'max_epochs': -1,
        'accelerator': 'auto', 'precision': 'fp32',
        'log_every_n_steps': 10, 'enable_progress_bar': True,
        'num_sanity_val_steps': 0, 'enable_checkpointing': False,
        'logger': False, 'use_distributed_sampler': False,
        'max_steps': 250,
        'limit_train_batches': 50,       # kept as int — 1.0 breaks on Lhotse's infinite dataloader
        'gradient_clip_val': 1.0,        # caps any single spike, given the step-49 blowup we found
    },
    'exp_manager': {
        'exp_dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run',
        'create_tensorboard_logger': True,
        'create_checkpoint_callback': True,
        'checkpoint_callback_params': {'monitor': 'val_wer', 'mode': 'min', 'save_top_k': 1, 'always_save_nemo': True},
        'resume_if_exists': False,
        'resume_ignore_no_checkpoint': True,
    },
    'init_from_nemo_model': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo',
}

out_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(full_cfg, f, allow_unicode=True, sort_keys=False)

print("Saved:", out_path)
print("Using scheduler:", model_cfg['optim']['sched']['name'], "| d_model:", d_model)

Saved: /home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml
Using scheduler: NoamAnnealing | d_model: 1024


!python "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr/asr_transducer/speech_to_text_rnnt_bpe_prompt.py" \
  --config-path="/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run" \
  --config-name="finetune_config"

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

ea = event_accumulator.EventAccumulator('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/default/<new_run_timestamp>')
ea.Reload()
print(ea.Tags()['scalars'])

loss_events = ea.Scalars('train_loss')
for e in loss_events:
    print(e.step, e.value)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-07-15 10:38:10 nemo_logging:364] /home/nextai/asr-nepali/Nepali-ASR-FineTuning/.myvenv/lib/python3.12/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
    See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
      ret = run_job(
    
[NeMo I 2026-07-15 10:38:10 speech_to_text_rnnt_bpe_prompt:78] Hydra config: name: FastConformer-Transducer-BPE-Prompt-Streaming-Pilot
    model:
      sample_rate: 16000
      compute_eval_loss: false
      log_prediction: true
      skip_nan_grad: true
      model_defaults:
        enc_hidden: 1024
        pred_hidden: 640
        j

In [5]:
import tarfile
import torch
import io

nemo_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

with tarfile.open(nemo_path, "r") as tar:
    f = tar.extractfile("model_weights.ckpt")
    ckpt = torch.load(io.BytesIO(f.read()), map_location="cpu")

print(type(ckpt))
print(ckpt.keys())

<class 'collections.OrderedDict'>
odict_keys(['preprocessor.featurizer.window', 'preprocessor.featurizer.fb', 'encoder.pre_encode.out.weight', 'encoder.pre_encode.out.bias', 'encoder.pre_encode.conv.0.weight', 'encoder.pre_encode.conv.0.bias', 'encoder.pre_encode.conv.2.weight', 'encoder.pre_encode.conv.2.bias', 'encoder.pre_encode.conv.3.weight', 'encoder.pre_encode.conv.3.bias', 'encoder.pre_encode.conv.5.weight', 'encoder.pre_encode.conv.5.bias', 'encoder.pre_encode.conv.6.weight', 'encoder.pre_encode.conv.6.bias', 'encoder.layers.0.norm_feed_forward1.weight', 'encoder.layers.0.norm_feed_forward1.bias', 'encoder.layers.0.feed_forward1.linear1.weight', 'encoder.layers.0.feed_forward1.linear2.weight', 'encoder.layers.0.norm_conv.weight', 'encoder.layers.0.norm_conv.bias', 'encoder.layers.0.conv.pointwise_conv1.weight', 'encoder.layers.0.conv.depthwise_conv.weight', 'encoder.layers.0.conv.batch_norm.weight', 'encoder.layers.0.conv.batch_norm.bias', 'encoder.layers.0.conv.pointwise_conv

In [6]:
state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

found = False
for key in state_dict.keys():
    if "prompt_kernel" in key:
        print(key)
        found = True

print("\nFound:", found)

prompt_kernel.0.weight
prompt_kernel.0.bias
prompt_kernel.2.weight
prompt_kernel.2.bias

Found: True


In [8]:
import torch

nan_params = []

for name, p in model.named_parameters():
    if torch.isnan(p).any():
        nan_params.append(name)

print("Number of parameters containing NaNs:", len(nan_params))

if nan_params:
    print("\nFirst few parameters with NaNs:")
    for name in nan_params[:20]:
        print(name)

NameError: name 'model' is not defined

In [4]:
import torch

ckpt = torch.load("model_weights.ckpt", map_location="cpu")

for key in ckpt["state_dict"]:
    if "prompt_kernel" in key:
        print(key)

FileNotFoundError: [Errno 2] No such file or directory: 'model_weights.ckpt'

!python "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr/asr_transducer/speech_to_text_rnnt_bpe_prompt.py" \
  --config-path="/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run" \
  --config-name="finetune_config" 2>&1 | tee /home/nextai/asr-nepali/Nepali-ASR-FineTuning/full_run_capture.txt

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-07-15 11:02:52 nemo_logging:364] /home/nextai/asr-nepali/Nepali-ASR-FineTuning/.myvenv/lib/python3.12/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
    See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
      ret = run_job(
    
[NeMo I 2026-07-15 11:02:52 speech_to_text_rnnt_bpe_prompt:78] Hydra config: name: FastConformer-Transducer-BPE-Prompt-Streaming-Pilot
    model:
      sample_rate: 16000
      compute_eval_loss: false
      log_prediction: true
      skip_nan_grad: true
      model_defaults:
        enc_hidden: 1024
        pred_hidden: 640
        j

: 

In [21]:
!wc -l /home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json

882 /home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json


In [ ]:
import json

dur = 0
count = 0
with open("/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json") as f:
    for line in f:
        obj = json.loads(line)
        dur += obj["duration"]
        count += 1

print("Files:", count)
print("Total hours:", dur/3600)
print("Average duration:", dur/count)

Files: 882
Total hours: 1.1958057638888886
Average duration: 4.880839852607709


: 

In [1]:
from tensorboard.backend.event_processing import event_accumulator

ea = event_accumulator.EventAccumulator('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/default/2026-07-15_08-36-30')
ea.Reload()

print(ea.Tags()['scalars'])  # see what metrics were logged (train_loss, val_loss, val_wer, etc.)

['hp_metric', 'train_loss', 'learning_rate', 'global_step', 'train_backward_timing in s', 'train_step_timing in s', 'training_batch_wer', 'epoch', 'validation_step_timing in s', 'val_wer', 'test_step_timing in s', 'test_wer']


In [2]:
loss_events = ea.Scalars('train_loss')
for e in loss_events:
    print(e.step, e.value)

9 196.87774658203125
19 239.3734588623047
29 274.2624206542969
39 207.35464477539062
49 806.344970703125
59 3000.848388671875
69 572.5271606445312
79 2502.853759765625
89 2000.7003173828125
99 107.29264068603516
109 2674.795654296875
119 409.7564392089844
129 162.67408752441406
139 2036.60986328125
149 797.3312377929688
159 3095.693359375
169 368.57086181640625
179 2720.8984375
189 181.34567260742188
199 164.6162872314453
209 238.5308380126953
219 367.07440185546875
229 282.2698669433594
239 2558.4658203125
249 959.2521362304688
259 2718.5556640625
269 286.55242919921875
279 160.4422149658203
289 133.73350524902344
299 916.0242309570312
309 232.2728271484375
319 400.8750915527344
329 2339.30908203125
339 2275.959228515625
349 864.981689453125
359 3149.068115234375
369 285.8661193847656
379 250.02447509765625
389 2497.855712890625
399 924.146484375
409 227.7764129638672
419 262.3798828125
429 186.0498809814453
439 2756.309814453125
449 620.814208984375
459 214.22450256347656
469 204.233

In [37]:
import json

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json', encoding='utf-8') as f:
    lines = f.readlines()

bad = []
for i, line in enumerate(lines):
    entry = json.loads(line)
    val = entry.get('target_lang')
    if not val or not isinstance(val, str) or not val.strip():
        bad.append((i, entry.get('audio_filepath'), repr(val)))

print('Total lines:', len(lines))
print('Bad entries:', len(bad))
for b in bad[:10]:
    print(b)

Total lines: 882
Bad entries: 0


In [38]:
import soundfile as sf

bad_audio = []
for line in lines:
    entry = json.loads(line)
    path = entry['audio_filepath']
    try:
        info = sf.info(path)
        if info.frames == 0:
            bad_audio.append(path)
    except Exception as e:
        bad_audio.append((path, str(e)))

print('Bad/unreadable audio files:', len(bad_audio))
print(bad_audio[:10])

Bad/unreadable audio files: 0
[]


In [ ]:
!awk '/^  train_ds:/,/^  validation_ds:/' ~/asr-nepali/Nepali-ASR-FineTuning/finetune_config.yaml | grep -i "lang_field\|prompt_field"

In [34]:
import json
paths = [json.loads(l)['audio_filepath'] for l in lines]
print('Duplicates:', len(paths) - len(set(paths)))

Duplicates: 0


## Testing (Not tested)

In [ ]:
manifest_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/pilot_manifest.json"
test_manifest_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json"

with open(manifest_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        if entry["audio_filepath"].endswith("/1025.wav"):
            with open(test_manifest_path, 'w', encoding='utf-8') as out:
                out.write(json.dumps(entry, ensure_ascii=False) + '\n')

            print(entry)
            break

In [ ]:
with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [ ]:
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt

finetuned_model = EncDecRNNTBPEModelWithPrompt.restore_from(
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/pilot_run/default/checkpoints/default.nemo"
)
finetuned_model = finetuned_model.cuda().eval()

In [ ]:
import lightning.pytorch as pl
from omegaconf import OmegaConf

finetuned_model.cfg.test_ds.manifest_filepath = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json"
finetuned_model.setup_test_data(finetuned_model.cfg.test_ds)

trainer = pl.Trainer(accelerator="gpu", devices=1, logger=False)
trainer.test(finetuned_model)